# I. The Building Blocks of Transformer Models

## 1. Transformers with PyTorch

### The paper that changed everything...

The paper, Attention Is All You Need (2017). 

Now, transformers are ubiquitous in deep learning, kickstarting the generative AI boom by forming the base for modern large language models, or LLMs. 

The transformer architecture is made of two blocks:
- the encoder block
- and the decoder block.

![image](images/image.png)

### The Encoder Block

The encoder block consists of multiple identical layers that are responsible for reading and processing the entire input sequence, generating context-rich numerical representations. 

It does this using self-attention and feed-forward networks, which we'll discuss in a moment.

![image-1](images/image-1.png)

### The Decoder Block

The decoder block essentially does the inverse of the encoder block, generating an output sequence based on the encoded input sequence.

### Positional encoding

Key to both encoder and decoder blocks is positional, which allows tokens to be processed in parallel by encoding each token's position in the sequence. 

This enables the model to recognize the relationships between tokens and their order, essential for making sense of sentences and capturing their context.

### Attention mechanisms

Attention mechanisms are used to highlight the most important tokens and their relationships, which improves the quality of generated text.

### Self-attention

Self-attention is a type of attention mechanism that assigns a weight to each token in the sequence simultaneously, capturing long-range dependencies.

### Multi-head attention

Multi-head attention extends self-attention by using multiple "heads" to focus on different aspects of the input sequence in parallel. 

This allows each head to capture distinct relational patterns within the data, leading to richer representations that enhance LLMs' effectiveness across tasks.

![image-2](images/image-2.png)


### Position-wise feed-forward networks
Finally, we have position-wise feed-forward networks. 

These are simple neural networks that apply complex transformations on each token's embeddings independently.

Because each token gets its own transformation, the networks are position-independent, hence, the "position-wise". 

### Transformers in PyTorch

PyTorch provides a high-level class in torch.nn to quickly define an architecture. 

nn.Transformer() takes four key parameters: 
- d_model, the dimensionality of the embedded sequence,
- the number of attention heads,
- and the number of encoder and decoder layers to include in the respective blocks. 

In [1]:
import torch.nn as nn

# Define the transformer model
model = nn.Transformer(
    d_model=1536,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6
)

# Print the model object
print(model)

Transformer(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, o

### Viewing the model

The output is pretty long. Inside the Transformer, we can actually see the transformer encoder block containing six transformer encoder layers, along with multi-head attention, and other bits you may have encountered elsewhere in deep learning, like linear layers, dropout layers, and layer normalization. 

Further down, we can also see the decoder block with six transformer decoder layers.


## 2. Embedding and positional encoding

### Embedding and positional encoding in transformers

The transformer architecture starts with **embedding sequences as vectors**, 

and then **encoding each token's position in the sequence** so that tokens can be **processed in parallel**.

---

### Embedding sequences

Suppose our input sequence has three tokens.

Each token has a unique ID in the model **vocabulary**, which is the set of tokens that the model can recognize.

When we embed them using an embedding layer, we get an embedding vector. 

The length of this vector is also referred to as the number of dimensions, or dimensionality. 
___
![image-3](images/image-3.png)

### Creating an embedding class

In the __init__ method, we define 

d_model, the dimensionality of the input embeddings, 

the model vocabulary size, 

and an embedding layer to map each token in the vocabulary to a vector. 

The forward method calculates and returns the embeddings. 

Scaling the embeddings by the square root of the dimensionality is a standard practice that ensures the token embeddings don't overwhelm or be overwhelmed by the positional embeddings.

In [2]:
import torch.nn as nn
import math

class InputEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, d_model: int) -> None:
        super().__init__()
        # Set the model dimensionality and vocabulary size
        self.d_model = d_model
        self.vocab_size = vocab_size
        # Instantiate the embedding layer
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        # Return the embeddings multiplied by the square root of d_model
        return self.embedding(x) * math.sqrt(self.d_model)

### Creating embeddings

Let's define an embedding layer with dimensionality 512 and vocab_size 10000. 

Passing an example batch of two sequences each with four token IDs into the embedding layer and printing the shape shows embeddings of the correct dimensionality for each token ID and batch. ``[2, 4, 512]``

In [7]:
import torch
token_ids = torch.tensor([[1, 2, 3, 4],
        [5, 6, 7, 8]])

# Instantiate InputEmbeddings and apply it to token_ids
embedding_layer = InputEmbeddings(10000, 512)
token_embeddings = embedding_layer(token_ids)
print(output.shape)

torch.Size([2, 4, 512])


### Positional encoding

Let's now discuss positional encoding. This encodes each token's position in the sequence into a positional embedding and adds them to the token embeddings to capture the positional information. 

The token and positional embeddings usually have the same dimensionality for easier addition.

These positional embeddings are generated using an equation that uses the token's position, and the sin and cosine mathematical functions, to generate unique positional embeddings. 

Sin is used for even embedding values, and cosine is used for odd values.

---

![image-4](images/image-4.png)

---

### Sin and cosine

For a token of a given position, its positional embedding vector can be calculated from these functions. 

The first value of the positional embedding vector, i=0, is calculated using the sin equation with i=0, as it is even. 

The second value, i=1, uses the cosine equation, and so on. 

d_model is the dimensionality, which is the same as the token embedding dimensionality. 

---

![image-5](image-5.png)

### Building a positional encoder

We define a PositionalEncoding class and initialize the positional embeddings, pe, to zeros. 

Next, we create a tensor of positions for each token in the sequence, which we'll transform using unsqueeze so it can be used in positional encoding calculations. 

Here are the sin and cosine calculations, matching the equations we just saw. The important thing to remember is that sin is applied to even vector values and cosine to odd values. 

register_buffer stores pe without making it a learnable parameter during training. 

Finally, we add the positional embeddings to the input token embeddings, x.

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super().__init__()
        # Create a matrix of zeros of dimensions max_seq_length by d_model
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        # Perform the sine and cosine calculations
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # Ensure pe isn't a learnable parameter during training
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        # Add the positional embeddings to the token embeddings
        return x + self.pe[:, :x.size(1)]

### Creating positional encodings

Let's create a positional encoder for our token embeddings. 
Applying the layer to the embedded_output produces a tensor of the same shape as the token embeddings.

In [8]:
pos_encoding_layer = PositionalEncoding(d_model=512, max_seq_length=4)
output = pos_encoding_layer(token_embeddings)
print(output.shape)
print(output[0][0][:10])

torch.Size([2, 4, 512])
tensor([-23.8086,  21.0167,  23.5505, -11.3023,  -5.1285, -34.7454,  31.3422,
         12.1421,   4.1019,  -2.9592], grad_fn=<SliceBackward0>)


## 3. Multi-head self-attention

### Self-attention mechanism

Self-attention enables transformers to identify relationships between tokens and focus on the most relevant ones for the task. 

Given a sequence of token embeddings, each embedding is projected onto three matrices of **equal dimensions** - query, key, and values - using separate linear transformations with learned weights. 

- Query indicates what each token is "looking for" in other tokens;
- key represents the "content" of each token that other tokens might find relevant, 
- values contains the actual content to be aggregated or weighted, based on the attention scores.

Transforming each token's embeddings into these roles helps the model learn more nuanced token relationships.

To obtain a matrix of attention scores between tokens, we compute the similarity between Q and K, typically using the dot-product metric.

Attention weights are calculated by applying softmax to the attention scores. 

---
![image-6](images/image-6.png)

---

The attention weights reflect the relevance or attention the model assigns to each token in a sequence. 

In the sequence,"orange is my favorite fruit," the tokens "favorite" and "fruit" receive the highest attention when processing "orange," as they directly influence its context and meaning. 

The model interprets "orange" as a favored fruit rather than a color or other meaning.

Finally, we multiply the values matrix, which are the token embeddings, by the attention weights to update the embeddings with the self-attention information. 

This self-attention mechanism uses one attention head with one set of Q, K, and V matrices.

---
![image-7](image-7.png)

### Multi-head attention

In practice, embeddings are split between multiple attention heads to focus on different aspects of the sequence in parallel, like relationships, sentiment, or subject.

Multi-head attention concatenates attention-head outputs, linearly transforming them to match the input dimensions. 

The resulting embeddings capture token meaning, positional encoding, and contextual relationships.

---

![image-8](images/image-8.png)

### MultiHeadAttention class

In the __init__ method, num_heads is the number of attention heads, and head_dim is the embedding dimensions each head will process. 

d_model must be divisible by num_heads. 

Three linear layers are defined for the attention inputs: query, key, and value, and one for the final concatenated output. 

Using bias=False for query, key, and value layers eliminates the bias term, reducing model parameters without impacting the ability to capture relationships.

```python
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads."
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model // num_heads

        self.query_linear = nn.Linear(d_model, d_model, bias=False)
        self.key_linear = nn.Linear(d_model, d_model, bias=False)
        self.value_linear = nn.Linear(d_model, d_model, bias=False)
        self.output_linear = nn.Linear(d_model, d_model)
```


We'll also define three helper methods for the different processes in the mechanism. 

split_heads splits the query, key, and values tensors between the heads and transforms them into shape: batch_size, seq_length, num_heads, head_dim. 

Inside compute_attention, torch.matmul calculates the dot product between the query and key matrices, which requires transposing the key matrix, and calculates the attention weights inside each head using softmax. 

We also add a condition to allow for masking, which we'll discuss more in the next chapter. 

Finally, we return these attention_weights multiplied by the value matrix. 

combine_heads transforms the attention weights back into the original embedding shape.

```python
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads."
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model // num_heads

        self.query_linear = nn.Linear(d_model, d_model, bias=False)
        self.key_linear = nn.Linear(d_model, d_model, bias=False)
        self.value_linear = nn.Linear(d_model, d_model, bias=False)
        self.output_linear = nn.Linear(d_model, d_model)

    # split and transform the input embeddings between the attention heads
    def split_heads(self, x, batch_size):
        seq_length = x.size(1)
        # Split the input embeddings and permute
        x = x.reshape(batch_size, seq_length, self.num_heads, self.head_dim)
        return x.permute(0, 2, 1, 3)

    # calculate the scaled dot-product attention weights multiplied by the values matrix
    def compute_attention(self, query, key, value, mask=None):
        # Compute scaled dot-product attention
        scores = torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attention_weights = F.softmax(scores, dim=-1)
        return torch.matmul(attention_weights, value)

    # transform the attention weights back into the same shape as the input embeddings, x
    def combine_heads(self, x, batch_size):
        seq_length = x.size(1)
        # Combine heads back to (batch_size, seq_length, d_model)
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.view(batch_size, -1, self.d_model)
        # OR: return x.reshape(batch_size, seq_length, self.d_model)
```


In the forward method, we split the query, key, and value tensors across the heads and compute the attention weights. 

The weights are combined and passed through the output layer to obtain the updated token embeddings projected into the original dimensionality.

```python
    # call the other methods to pass the input embeddings through each process
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # Build the forward pass
        query = self.split_heads(self.query_linear(query), batch_size)
        key = self.split_heads(self.key_linear(key), batch_size)
        value = self.split_heads(self.value_linear(value), batch_size)
        
        attention_weights = self.compute_attention(query, key, value, mask)
        output = self.combine_heads(attention_weights, batch_size)
        return self.output_linear(output)
```

In [ ]:
# Define attention parameters
d_model = 512
num_heads = 8

# Instantiate a MultiHeadAttention instance
multihead_attn = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

# Pass the query, key, and value matrices through the mechanism
output = multihead_attn(query, key, value)
print(output.shape)

query.shape = torch.Size([16, 10, 512]) # (batch_size, seq_length, d_model)

output.shape = torch.Size([16, 10, 512])

matches the shape of the input embeddings, which you would expect as the multi-head attention mechanism is essentially adding more information to the input embeddings.

# II. Building Transformer Architectures

## 1. Encoder transformers

### The original transformer

The transformer architecture combines an encoder and decoder for sequence-to-sequence language representation and generation.

The output from the encoder is inputted into the decoder layers to form the bridge between the two blocks.

---

![image-9](images/image-9.png)

### Encoder-only transformers

Encoder-only transformers simplify this architecture to place greater emphasis on **understanding and representing the input data, such as text classification**. 

They have two main components: the transformer body and head. 

The transformer body, or encoder, in this case, is a stack of multiple encoder layers designed to learn complex patterns from the inputs. 

Each encoder layer incorporates a multi-head self-attention mechanism to capture relationships between tokens in the sequence, followed by feed-forward sublayers to map this knowledge into abstract, nonlinear representations. 

Both elements are usually combined with other techniques like layer normalizations and dropouts to improve training.

---
![image-10](images/image-10.png)

---

The head is the final layer designed to produce task-specific outputs. 

In encoder transformers, they are typically supervised learning outputs like classification labels or regression predictions.

### Feed-forward sublayer in encoder layers

Our FeedForwardSublayer class contains two fully connected linear layers separated by a ReLU activation. 

Notice we use a dimension d_ff between linear layers, typically different from the embedding dimension used throughout the model to further facilitate capturing complex patterns. 

The forward method applies the forward pass to the attention mechanism outputs, passing them through the layers.

![image-11](images/image-11.png)

```python
class FeedForwardSubLayer(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # Define the layers and activation
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Pass the input through the layers and activation
        return self.fc2(self.relu(self.fc1(x)))
    
# Instantiate the FeedForwardSubLayer and apply it to x
d_model = 512
d_ff = 2048

feed_forward = FeedForwardSubLayer(d_model, d_ff)
output = feed_forward(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
```

### Encoder layer

We've identified two key elements for implementing an encoder layer, multi-head self-attention and feed-forward sublayers. 

These are incorporated into the encoder layer with layer normalizations for keeping the scales and variances of input embeddings consistent before and after the feed forward layer. 

Dropouts are also used to regularize and stabilize training. 

Notice when calling the attention mechanism in the forward pass the input embeddings are passed as the query, key, and value matrices, 

and a mask is used to prevent the processing of padding tokens in the input sequence. 

---

![image-12](images/image-12.png)

```python
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        # Instantiate the layers
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff_sublayer = FeedForwardSubLayer(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        # Complete the forward method
        attn_output = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.ff_sublayer(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x
```

### Masking the attention process

In NLP tasks with variable-length input sequences, padding ensures equal length across sequences by appending padding tokens, or zeros in this case. 

These padded tokens are irrelevant for the language task so we exclude them from the attention mechanism. 

We do this by applying a padding mask to the attention scores, so scores linked to padded tokens are set to zero.

---

![image-13](images/image-13.png)

### Encoder transformer body

Once we fully implement an encoder layer, we stack several layers together to build our transformer body: the encoder. 

We define the token embedding based on the vocabulary size, followed by positional encoding using the previously-defined class, and a stack of multiple encoder layers, using PyTorch's ModuleList class and a list comprehension. 

The forward pass embeds the inputs, performs positional encoding, and iterates through the encoder layers, applying the padding mask.

---

![image-15](image-15.png)


```python
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length):
        super().__init__()
        # Define the embedding, positional encoding, and encoder layers
        self.embedding = InputEmbeddings(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

    def forward(self, x, src_mask):
        # Perform the forward pass through the layers
        x = self.embedding(x)
        x = self.positional_encoding(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return x
```

### Encoder transformer head

For the transformer head, we can create a classification head, suitable for tasks like text classification and sentiment analysis. 

It consists of a linear layer with softmax activation to map the resulting encoder hidden states into class probabilities. 

A regression head has a linear layer with an output dimension equal to the number of target regression outputs.

It's suited for tasks like estimating text readability or language complexity.

--- 

![image-14](images/image-14.png)


```python
vocab_size = 10_000
d_model = 512
num_layers = 6
num_heads = 8
d_ff = 2048
dropout = 0.1
seq_length = 256
num_classes = 3
input_sequence.size() = torch.Size([8, 256])
src_mask.size() = torch.Size([256, 256])
src_mask = """
    tensor([[0, 1, 0,  ..., 0, 0, 0],
            [0, 1, 1,  ..., 1, 0, 0],
            [0, 0, 0,  ..., 1, 0, 1],
            ...,
            [1, 1, 0,  ..., 0, 0, 1],
            [0, 0, 1,  ..., 1, 1, 0],
            [0, 1, 0,  ..., 0, 0, 1]])
"""
```

```python
# Complete the classification head
class ClassifierHead(nn.Module):
    def __init__(self, d_model, num_classes):
        super().__init__()
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        logits = self.fc(x)
        return F.log_softmax(logits, dim=-1)
      
# Instantiate the encoder transformer's body and head
encoder = TransformerEncoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, seq_length)
classifier = ClassifierHead(d_model, num_classes)

# Complete the forward pass 
output = encoder(input_sequence, src_mask)
classification = classifier(output)
print(f"Classification outputs for a batch of {batch_size} sequences:\n{classification}")
print(f"Encoder output shape: {output.shape}\nClassification head output shape: {classification.shape}")
```

```bash
Encoder output shape: torch.Size([8, 256, 512])
Classification head output shape: torch.Size([8, 256, 3])
```

> Until trained these predictions are meaningless, but it's a good test that the class works as expected!

## 2. Decoder transformers

### From original to decoder-only transformer

The decoder-only architecture is again a simplified version of the original transformer, specialized in sequence generation tasks not requiring an encoder. 

Concretely, it is designed to handle autoregressive sequence generation tasks like text generation and completion, where each token in the sequence is predicted based only on the preceding tokens. 

The architecture is similar to an encoder-only approach, with two differences.

### Masked multi-head self-attention

One is the use of masked multi-head self-attention, which masks future tokens in the sequence to enable the model to learn and predict these future tokens using only the prior tokens. 

Otherwise, the model would be able to "look ahead" and cheat rather than learning to predict. 

For each token in the target sequence, only the previously generated tokens are observed, whereas subsequent tokens are hidden using a causal attention mask.

---
![image-17](images/image-17.png)


### Transformer head
The other difference lies in the transformer head, which generally consists of a linear layer with softmax activation over the entire vocabulary to estimate the likelihood of each token being the next one in the sequence, and sampling a token from the most likely ones.

---

![image-16](images/image-16.png)

### Masked self-attention/causal attention

Let's explore and implement these two elements, starting with masked self-attention, also called causal attention. 

This is key to giving our model an **autoregressive or causal behavior**, and it is achieved by using an upper triangular or causal attention mask as shown.

By passing this matrix to the attention heads, each token only pays attention to the tokens that came before it in the sequence.


For instance, during training, the token "favorite" in the sequence "orange is my favorite fruit" would only pay attention to the previous tokens: orange, is, my, and favorite. 

This way, during inference, the model will predict the likelihood of the next token, and should assign a relatively high probability to "fruit". 

To create the mask, we create a matrix of ones of shape sequence length by sequence length using torch.ones. 

Then, we torch.triu to make all elements of this matrix below the diagonal zero, subtract this matrix from one to get zero above the diagonal and minus one below it, and convert to bools, so zero values are False and non-zero values are True.

---

![image-18](images/image-18.png)


```python
seq_length= 3

# Create a Boolean matrix to mask future tokens
tgt_mask = (1 - torch.triu(
  torch.ones(1, seq_length, seq_length), diagonal=1)
).bool()

print(tgt_mask)
```

### Decoder layer

The decoder layer is the same as the encoder layer we created before: multi-head attention followed by a feed-forward sublayer with layer normalizations and dropouts before and after. 

The only difference is the padding mask has been replaced with the causal attention mask in the forward pass.

![image-19](images/image-19.png)

```python
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff_sublayer = FeedForwardSubLayer(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, tgt_mask):
        # Perform the attention calculation
        attn_output = self.self_attn(x, x, x, tgt_mask)
        # Apply dropout and the first layer normalization
        x = self.norm1(x + self.dropout(attn_output))
        # Pass through the feed-forward sublayer
        ff_output = self.ff_sublayer(x)
        # Apply dropout and the second layer normalization
        x = self.norm2(x + self.dropout(ff_output))
        return x
```

### Decoder transformer body and head

Regarding the transformer head, it can be implemented outside or inside the transformer body class. 

Let's illustrate how to add the head inside the decoder transformer class itself. 

Most of the code is identical to the encoder-only transformer class seen previously. 

We embed the inputs, create the positional encodings, and pass the input through the decoder layers. 

We add a final linear layer with output size equal to the vocabulary size, and softmax activation to the forward method, to project hidden states into word likelihoods.

![image-20](images/image-20.png)

```python
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length):
        super(TransformerDecoder, self).__init__()
        self.embedding = InputEmbeddings(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        # Define the list of decoder layers and linear layer
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        # Define a linear layer to project hidden states to likelihoods
        self.fc = nn.Linear(d_model, vocab_size)
  
    def forward(self, x, tgt_mask):
        # Complete the forward pass
        x = self.embedding(x)
        x = self.positional_encoding(x)
        for layer in self.layers:
            x = layer(x, tgt_mask)
        x = self.fc(x)
        return F.log_softmax(x, dim=-1)
```

### Instantiating the decoder-only transformer

After we instantiate the decoder, we can pass it the causal mask to be used in the attention mechanism.

![image-21](images/image-21.png)


```python
# Instantiate a decoder transformer and apply it to input_tokens and tgt_mask
transformer_decoder = TransformerDecoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length)   
output = transformer_decoder(input_tokens, tgt_mask)
print(output)
print(output.shape)
```

The shape [2, 10, 10000] matches [batch_size, seq_length, vocab_size], so your decoder transformer successfully projected the embeddings into token probabilities. 

Remember, because this model hasn't been trained, these probabilities won't be meaningful.

## 3. Encoder-decoder transformers

### Encoder meets decoder

Let's build a full encoder-decoder transformer for sequence-to-sequence language tasks like text translation.

To connect our encoder and decoder bodies, we need to direct the encoder outputs into the decoder.

---
![image-22](images/image-22.png)

---

![image-23](images/image-23.png)

---

To do this, we need one more component: a cross-attention mechanism.

### Cross-attention mechanism

Cross-attention is another variant of the self-attention mechanism. 

It occurs **in each decoder layer after the masked attention**, taking two inputs: the information processed through the decoder, and the final hidden states produced by the encoder, thereby linking the two transformer blocks. 

This is crucial for the decoder to "look back" at the input sequence to figure out what to generate next in the target sequence. 

Consider this English-to-Spanish translation example. Given the sequence "I really like to travel", cross-attention identifies key words in the original English sequence to generate the next Spanish word: in this case, "travel" receives the highest attention.

---

![image-24](images/image-24.png)

### Modifying the DecoderLayer

This additional attention stage is implemented inside the decoder layer class using the same MultiHeadAttention class we created. 

The forward() method now requires two masks: the causal mask for the first attention stage, and the cross-attention mask, which can be the same padding mask used in the encoder. 

Importantly, **the variable y in this method represents the encoder outputs**, passed as **key and value arguments** to the cross-attention mechanism. 

Meanwhile, the decoder flow, x, associated with generating the target sequence, now only acts as the attention query. 

After cross-attention in the forward pass, we pass through the feed-forward sublayers as normal.

---
![image-25](images/image-25.png)

```python
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        # Define cross-attention and a third layer normalization
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ff_sublayer = FeedForwardSubLayer(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, y, tgt_mask, cross_mask):
        self_attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_output))
        # Complete the forward pass
        cross_attn_output = self.cross_attn(x, y, y, cross_mask)
        x = self.norm2(x + self.dropout(cross_attn_output))
        ff_output = self.ff_sublayer(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x
```

### Modifying DecoderTransformer

Because we altered the DecoderLayer class to add the cross-attention mask, we also need to make similar modifications to the forward method of the **TransformerDecoder class**, passing the encoder outputs, y, and cross_mask to each decoder layer in addition to x and the causal mask.

---
![image-26](images/image-26.png)

The final encoder outputs are now fed into every layer of the decoder for cross-attention.

---
![image-28](image-28.png)

### Transformer head

**Similar to decoder-only transformers**, the model's output head consists of a linear layer followed by softmax activation, converting decoder outputs into next-word probabilities. 

In our translation example, these are probabilities for different Spanish words to be generated next. 

In other language tasks, a different activation function instead of softmax may be required.

![image-29](images/image-29.png)

### Everything brought together!

Here is the full transformer architecture we just learned to build. 

---
![image-30](images/image-30.png)

--- 
Let's use the classes we've built throughout the course to create an encoder-decoder Transformer class.

With all our classes defined to build the layer components, the layers themselves, and the transformer blocks, we can create a Transformer class that encapsulates everything. 

We initialize the transformer encoder and decoder stacks, and define a two-stage forward pass: passing the input sequence, x, through the encoder, and then passing it to the decoder together with the encoder output, incorporating the cross-attention mask.

![image-31](images/image-31.png)


```python
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length)
        self.decoder = TransformerDecoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_length)

    def forward(self, x, src_mask, tgt_mask, cross_mask):
        # Complete the forward pass
        encoder_output = self.encoder(x, src_mask)
        decoder_output = self.decoder(x, encoder_output, tgt_mask, cross_mask)
        return decoder_output

# Instantiate and call the transformer
transformer = Transformer(vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)
outputs = transformer(input_tokens, src_mask, tgt_mask, cross_mask)
print(outputs)
print(outputs.shape)
```